# 02 — CU2: fine-tuning de BETO

Ajusta `dccuchile/bert-base-spanish-wwm-cased` (BETO) a las 21
categorías, con early stopping sobre el F1-macro de validación. Meta
(OE2): F1-macro ≥ 0,85 en el conjunto de prueba, y superar al baseline
de 01_cu2_baseline_tfidf.

Tiempo esperado en GPU T4: 15-30 minutos para el dataset completo (3-4
épocas). En CPU (MODO_RAPIDO) es cuestión de minutos porque solo corre 1
época sobre un puñado de ejemplos.

In [ ]:
# Si ya corriste esta celda en esta sesión de Colab, puedes saltarla.
import sys

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_ML = "/content/drive/MyDrive/NovaStore_IA/ml"
    get_ipython().system(f"pip install -q -r {{CARPETA_ML}}/requirements.txt")
else:
    # Ejecución local (verificación, no Colab): usa la carpeta ml/ del repo.
    CARPETA_ML = str(__import__("pathlib").Path.cwd())

sys.path.insert(0, CARPETA_ML)
print("CARPETA_ML =", CARPETA_ML)


In [ ]:
import random
from pathlib import Path

import numpy as np
import torch

# --- Configuración: edita esto antes de correr en Colab ---
CARPETA_DRIVE = Path(CARPETA_ML) / "datos"  # train.csv/val.csv/test.csv (los crea 00_preparar_dataset.ipynb)
CARPETA_MODELOS = Path(CARPETA_ML) / "modelos" / "cu2-beto-v1"
MODO_RAPIDO = False  # True = 5 ejemplos/clase, 1 época — solo para probar que el notebook corre de punta a punta
SEMILLA = 42
# ------------------------------------------------------------

random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print(f"MODO_RAPIDO={MODO_RAPIDO} · dispositivo={DISPOSITIVO} · GPU disponible: {torch.cuda.is_available()}")
if DISPOSITIVO == "cpu":
    print("Sin GPU: en Colab, activa Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU.")

CARPETA_MODELOS.mkdir(parents=True, exist_ok=True)


## 1. Cargar la partición

In [ ]:
import pandas as pd

if MODO_RAPIDO:
    from _modo_rapido_fallback import generar_si_falta
    generar_si_falta(CARPETA_DRIVE)


train = pd.read_csv(CARPETA_DRIVE / "train.csv")
val = pd.read_csv(CARPETA_DRIVE / "val.csv")
test = pd.read_csv(CARPETA_DRIVE / "test.csv")

if MODO_RAPIDO:
    train = train.groupby("categoria", group_keys=False).head(5)
    val = val.groupby("categoria", group_keys=False).head(2)
    test = test.groupby("categoria", group_keys=False).head(2)

print(f"train={len(train)} val={len(val)} test={len(test)}")
train.head()


## 2. Tokenizar y armar los DataLoaders

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

MODELO_BASE = "dccuchile/bert-base-spanish-wwm-cased"
CLASES = sorted(train["categoria"].unique())
LABEL2ID = {c: i for i, c in enumerate(CLASES)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}

class DatasetTexto(Dataset):
    def __init__(self, textos, etiquetas, tokenizer):
        self.encodings = tokenizer(list(textos), truncation=True, max_length=128, padding=True)
        self.etiquetas = list(etiquetas)

    def __len__(self):
        return len(self.etiquetas)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.etiquetas[idx])
        return item

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
y_train = train["categoria"].map(LABEL2ID)
y_val = val["categoria"].map(LABEL2ID)
y_test = test["categoria"].map(LABEL2ID)

BATCH_SIZE = 16
train_loader = DataLoader(DatasetTexto(train["texto_es"], y_train, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(DatasetTexto(val["texto_es"], y_val, tokenizer), batch_size=BATCH_SIZE)
test_loader = DataLoader(DatasetTexto(test["texto_es"], y_test, tokenizer), batch_size=BATCH_SIZE)

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE, num_labels=len(CLASES), id2label=ID2LABEL, label2id=LABEL2ID
).to(DISPOSITIVO)
print(f"{len(CLASES)} clases · {len(train)} train / {len(val)} val / {len(test)} test")


## 3. Entrenar con early stopping sobre F1-macro de validación

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

EPOCAS = 1 if MODO_RAPIDO else 4
LR = 2e-5

optimizador = torch.optim.AdamW(modelo.parameters(), lr=LR)
total_pasos = len(train_loader) * EPOCAS
scheduler = get_linear_schedule_with_warmup(optimizador, 0, total_pasos)

def evaluar(loader):
    modelo.eval()
    y_true, y_pred, perdida_total = [], [], 0.0
    with torch.no_grad():
        for lote in loader:
            etiquetas = lote.pop("labels")
            lote = {k: v.to(DISPOSITIVO) for k, v in lote.items()}
            etiquetas = etiquetas.to(DISPOSITIVO)
            salida = modelo(**lote, labels=etiquetas)
            perdida_total += salida.loss.item() * len(etiquetas)
            y_pred.extend(salida.logits.argmax(dim=1).cpu().tolist())
            y_true.extend(etiquetas.cpu().tolist())
    return {
        "loss": perdida_total / max(len(y_true), 1),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

historia = {"epoca": [], "train_loss": [], "val_loss": [], "val_accuracy": [], "val_f1_macro": []}
mejor_f1, mejor_estado, epocas_sin_mejora = -1.0, None, 0
PACIENCIA = 2

for epoca in range(1, EPOCAS + 1):
    modelo.train()
    perdida_total = 0.0
    for lote in train_loader:
        etiquetas = lote.pop("labels").to(DISPOSITIVO)
        lote = {k: v.to(DISPOSITIVO) for k, v in lote.items()}
        optimizador.zero_grad()
        salida = modelo(**lote, labels=etiquetas)
        salida.loss.backward()
        optimizador.step()
        scheduler.step()
        perdida_total += salida.loss.item() * len(etiquetas)

    metricas_val = evaluar(val_loader)
    historia["epoca"].append(epoca)
    historia["train_loss"].append(perdida_total / len(train))
    historia["val_loss"].append(metricas_val["loss"])
    historia["val_accuracy"].append(metricas_val["accuracy"])
    historia["val_f1_macro"].append(metricas_val["f1_macro"])
    print(f"época {epoca}/{EPOCAS}  train_loss={historia['train_loss'][-1]:.4f}  val_f1_macro={metricas_val['f1_macro']:.4f}")

    if metricas_val["f1_macro"] > mejor_f1:
        mejor_f1 = metricas_val["f1_macro"]
        mejor_estado = {k: v.cpu().clone() for k, v in modelo.state_dict().items()}
        epocas_sin_mejora = 0
    else:
        epocas_sin_mejora += 1
        if epocas_sin_mejora >= PACIENCIA:
            print(f"Early stopping en la época {epoca} (sin mejora hace {PACIENCIA} épocas).")
            break

modelo.load_state_dict(mejor_estado)
print(f"Mejor F1-macro de validación: {mejor_f1:.4f}")


## 4. Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(historia["epoca"], historia["train_loss"], label="train")
ax1.plot(historia["epoca"], historia["val_loss"], label="val")
ax1.set_title("Pérdida"); ax1.set_xlabel("época"); ax1.legend()
ax2.plot(historia["epoca"], historia["val_accuracy"], label="val accuracy")
ax2.plot(historia["epoca"], historia["val_f1_macro"], label="val F1-macro")
ax2.set_title("Validación"); ax2.set_xlabel("época"); ax2.legend()
plt.tight_layout()
plt.savefig(CARPETA_MODELOS / "curvas_entrenamiento.png", dpi=120)
plt.show()


## 5. Evaluar sobre PRUEBA y comparar contra el baseline

In [ ]:
import json

from sklearn.metrics import classification_report, precision_score, recall_score

modelo.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for lote in test_loader:
        etiquetas = lote.pop("labels")
        lote = {k: v.to(DISPOSITIVO) for k, v in lote.items()}
        logits = modelo(**lote).logits
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())
        y_true.extend(etiquetas.tolist())

metricas = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
    "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
    # labels=range(len(CLASES)) a propósito: si el set de prueba no tiene
    # ejemplos de alguna clase (pasa fácil en MODO_RAPIDO, y puede pasar
    # en una corrida real con una clase muy chica), classification_report
    # infiere las clases de y_true/y_pred — con menos clases que CLASES,
    # target_names no cuadra y revienta. Fijar labels lo evita siempre.
    "reporte_por_clase": classification_report(
        y_true, y_pred, labels=list(range(len(CLASES))), target_names=CLASES, output_dict=True, zero_division=0
    ),
}
print(json.dumps({k: v for k, v in metricas.items() if k != "reporte_por_clase"}, indent=2))

ruta_baseline = CARPETA_ML_PATH = (CARPETA_MODELOS.parent / "cu2-tfidf-logreg-v1" / "metricas.json")
if ruta_baseline.exists():
    baseline = json.loads(ruta_baseline.read_text())
    tabla = {
        "modelo": ["TF-IDF + LogReg (baseline)", "BETO (fine-tuned)"],
        "f1_macro": [baseline["f1_macro"], metricas["f1_macro"]],
        "accuracy": [baseline["accuracy"], metricas["accuracy"]],
    }
    import pandas as pd
    print()
    print(pd.DataFrame(tabla).to_string(index=False))
    print("\nMeta OE2 (F1-macro>=0.85 y > baseline):",
          "CUMPLIDA" if metricas["f1_macro"] >= 0.85 and metricas["f1_macro"] > baseline["f1_macro"] else "NO alcanzada todavía")
else:
    print("(No se encontró metricas.json del baseline — corre 01_cu2_baseline_tfidf.ipynb primero para comparar.)")

with open(CARPETA_MODELOS / "metricas.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)


## 6. Matriz de confusión

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay.from_predictions(
    [CLASES[i] for i in y_true], [CLASES[i] for i in y_pred], xticks_rotation="vertical", ax=ax, colorbar=False
)
plt.tight_layout()
plt.savefig(CARPETA_MODELOS / "matriz_confusion.png", dpi=120)
plt.show()


## 7. Guardar modelo + función de embeddings

El embedding de CU5 (duplicados) sale de BETO con mean pooling sobre la
última capa oculta (768 dimensiones) — no del token [CLS] solo, que es
una variante más ruidosa para similitud semántica.

In [ ]:
modelo.save_pretrained(CARPETA_MODELOS)
tokenizer.save_pretrained(CARPETA_MODELOS)

def obtener_embedding(texto: str) -> list[float]:
    entrada = tokenizer(texto, return_tensors="pt", truncation=True, max_length=128, padding=True).to(DISPOSITIVO)
    with torch.no_grad():
        salida = modelo.base_model(**entrada)
        mascara = entrada["attention_mask"].unsqueeze(-1).float()
        suma = (salida.last_hidden_state * mascara).sum(1)
        conteo = mascara.sum(1).clamp(min=1e-9)
        embedding = (suma / conteo).squeeze(0)
    return embedding.cpu().tolist()

vector_prueba = obtener_embedding(test["texto_es"].iloc[0])
print(f"Embedding de prueba: {len(vector_prueba)} dimensiones (debe ser 768).")
assert len(vector_prueba) == 768

sql = f"""
UPDATE novastore.modelos_ia
   SET metricas = '{json.dumps({k: v for k, v in metricas.items() if k != "reporte_por_clase"})}'::jsonb,
       hiperparametros = hiperparametros || '{{"lr": {LR}, "epocas_corridas": {len(historia["epoca"])}, "max_length": 128}}'::jsonb,
       estado = 'evaluado',
       entrenado_en = now(),
       ruta_artefacto = 'cu2-beto-v1/'
 WHERE codigo = 'cu2-beto-v1';
"""
(CARPETA_MODELOS / "registro_modelo.sql").write_text(sql, encoding="utf-8")
print(sql)
